# Task 5: Benchmarking and Metrics

A repeatable precision/recall/F1 harness for the Task 2 extraction pipeline, scaled from a single probe document to a small labeled sample.

- **Dataset:** [DocRED](https://huggingface.co/datasets/thunlp/docred) `train_annotated` split (gold-labeled documents) — same source as Task 2.
- **Pipeline under test:** the same spaCy `en_core_web_sm` NER baseline vs. DocRED gold entities (all aliases) from Task 2, generalized into reusable functions and run across 25 documents instead of 1.
- **Why this matters:** Task 2 only ever reported a single number (17/22 — 77% — on one document), which was actually a *recall* figure, not precision or F1 (spaCy also produces false positives with no gold counterpart). This notebook computes true precision/recall/F1, aggregated across a sample, so future extraction or merging changes have something repeatable to compare against.
- **Flow:** load dataset → sample documents → build reusable scoring harness → run across the sample → aggregate (micro + macro) → cross-check against Task 2's single-document result → findings write-up (bottom of notebook).

### Step 1: Setup

In [1]:
%pip install datasets spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re
import random

import pandas as pd
import spacy

from datasets import load_dataset

nlp = spacy.load("en_core_web_sm")

### Step 2: Load DocRED and select the gold-annotated portion

Same fixes established in Task 2: load from the `refs/convert/parquet` revision (the legacy loading script no longer works), then explicitly slice to the first 3,053 rows to avoid the noisy `train_distant` documents merged into the same `train` split.

In [3]:
ds = load_dataset("thunlp/docred", revision="refs/convert/parquet")

NUM_ANNOTATED = 3053
train_annotated = ds["train"].select(range(NUM_ANNOTATED))
print(f"Using {len(train_annotated)} gold-annotated documents (train_annotated portion only)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


default/train_annotated/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.07MB            

default/train_annotated/0000.parquet: downloading bytes:           |  0.00B            

default/train_distant/0000.parquet: reconstructing file:   0%|          |  0.00B /  101MB            

default/train_distant/0000.parquet: downloading bytes:           |  0.00B            

...

0000.parquet:   0%|          | 0.00/943k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Using 3053 gold-annotated documents (train_annotated portion only)


### Step 3: Sample a small set of documents for benchmarking

Document 0 ("AirAsia Zest") is deliberately included — it's the exact probe document Task 2 reported 77% on, so it doubles as a direct cross-check between the two notebooks.

In [4]:
random.seed(0)

N_SAMPLE = 25
doc_indices = [0] + random.sample(range(1, NUM_ANNOTATED), N_SAMPLE - 1)
print(f"Benchmarking on {len(doc_indices)} documents (including doc 0, Task 2's probe document)")

Benchmarking on 25 documents (including doc 0, Task 2's probe document)


### Step 4: Reusable extraction + scoring harness (generalized from Task 2)

`normalize_entity` is unchanged from Task 2 (strips tokenization artifacts, leading articles, trailing possessives). `gold_names_from_doc` keeps Task 2's alias-blindness fix — every mention name per DocRED entity counts, not just the first.

In [5]:
def normalize_entity(name):
    name = name.lower().strip()
    name = re.sub(r"\s+([,.;:'])", r"\1", name)
    name = re.sub(r"^(the|a|an)\s+", "", name)
    name = re.sub(r"'s$", "", name)
    return name.strip()

def doc_text_from_doc(doc):
    return " ".join(" ".join(sent) for sent in doc["sents"])

def gold_names_from_doc(doc):
    names = set()
    for v in doc["vertexSet"]:
        for m in v:
            names.add(normalize_entity(m["name"]))
    return names

def baseline_names_from_text(text, nlp):
    spacy_doc = nlp(text)
    return {normalize_entity(ent.text) for ent in spacy_doc.ents}

def score_entities(gold_names, baseline_names):
    tp = len(gold_names & baseline_names)
    fp = len(baseline_names - gold_names)
    fn = len(gold_names - baseline_names)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return tp, fp, fn, precision, recall, f1

### Step 5: Run the harness across the sample

In [6]:
records = []
for idx in doc_indices:
    doc = train_annotated[idx]
    text = doc_text_from_doc(doc)
    gold_names = gold_names_from_doc(doc)
    baseline_names = baseline_names_from_text(text, nlp)
    tp, fp, fn, precision, recall, f1 = score_entities(gold_names, baseline_names)

    records.append({
        "doc_index": idx,
        "title": doc["title"],
        "n_gold": len(gold_names),
        "n_baseline": len(baseline_names),
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
    })
    print(f"doc {idx:>5} ({doc['title'][:40]:40}): P={precision:.2f} R={recall:.2f} F1={f1:.2f}  (gold={len(gold_names)}, baseline={len(baseline_names)})")

results_df = pd.DataFrame(records)
results_df

doc     0 (AirAsia Zest                            ): P=0.81 R=0.77 F1=0.79  (gold=22, baseline=21)
doc  1578 (Eickendorf, Salzlandkreis               ): P=0.79 R=0.76 F1=0.78  (gold=25, baseline=24)
doc  1723 (Victorian Gay and Lesbian Rights Lobby  ): P=0.47 R=0.60 F1=0.53  (gold=15, baseline=19)
doc   166 (Mount Cardigan                          ): P=0.71 R=0.92 F1=0.80  (gold=13, baseline=17)
doc  1061 (The Emperor's Bridge Campaign           ): P=0.77 R=0.62 F1=0.69  (gold=16, baseline=13)
doc  2095 (Poor Boy (The Greenwood)                ): P=0.63 R=0.57 F1=0.60  (gold=21, baseline=19)
doc  1991 (The C Word                              ): P=0.88 R=0.94 F1=0.91  (gold=32, baseline=34)
doc  1659 (Free Software Foundation                ): P=1.00 R=0.80 F1=0.89  (gold=10, baseline=8)
doc  1243 (Hobo Jim                                ): P=0.88 R=0.92 F1=0.90  (gold=24, baseline=25)
doc  1953 (Guitar Hero: Aerosmith                  ): P=0.81 R=0.81 F1=0.81  (gold=26, baseline=26)
d

,doc_index,title,n_gold,n_baseline,tp,fp,fn,precision,recall,f1
0,0,AirAsia Zest,22,21,17,4,5,0.809524,0.772727,0.790698
1,1578,"Eickendorf, Salzlandkreis",25,24,19,5,6,0.791667,0.760000,0.775510
2,1723,Victorian Gay and Lesbian Rights Lobby,15,19,9,10,6,0.473684,0.600000,0.529412
3,166,Mount Cardigan,13,17,12,5,1,0.705882,0.923077,0.800000
4,1061,The Emperor's Bridge Campaign,16,13,10,3,6,0.769231,0.625000,0.689655
5,2095,Poor Boy (The Greenwood),21,19,12,7,9,0.631579,0.571429,0.600000
6,1991,The C Word,32,34,30,4,2,0.882353,0.937500,0.909091
7,1659,Free Software Foundation,10,8,8,0,2,1.000000,0.800000,0.888889
8,1243,Hobo Jim,24,25,22,3,2,0.880000,0.916667,0.897959
9,1953,Guitar Hero: Aerosmith,26,26,21,5,5,0.807692,0.807692,0.807692


### Step 6: Aggregate metrics (micro vs. macro)

In [7]:
total_tp = results_df["tp"].sum()
total_fp = results_df["fp"].sum()
total_fn = results_df["fn"].sum()

micro_p = total_tp / (total_tp + total_fp)
micro_r = total_tp / (total_tp + total_fn)
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r)

macro_p = results_df["precision"].mean()
macro_r = results_df["recall"].mean()
macro_f1 = results_df["f1"].mean()

print(f"Micro-averaged (pooled TP/FP/FN across all {len(results_df)} docs): P={micro_p:.3f}  R={micro_r:.3f}  F1={micro_f1:.3f}")
print(f"Macro-averaged (mean of per-doc P/R/F1):                       P={macro_p:.3f}  R={macro_r:.3f}  F1={macro_f1:.3f}")

Micro-averaged (pooled TP/FP/FN across all 25 docs): P=0.788  R=0.788  F1=0.788
Macro-averaged (mean of per-doc P/R/F1):                       P=0.795  R=0.792  F1=0.790


### Step 7: Cross-check against Task 2's single-document result

In [8]:
doc0_row = results_df[results_df["doc_index"] == 0].iloc[0]
print("Doc 0 (Task 2's probe document, 'AirAsia Zest'):")
print(f"  gold={doc0_row['n_gold']}  baseline={doc0_row['n_baseline']}")
print(f"  precision={doc0_row['precision']:.3f}  recall={doc0_row['recall']:.3f}  f1={doc0_row['f1']:.3f}")
print("\nTask 2 reported '17/22 (77%) match' for this document — that figure is recall (matched / gold), not precision or F1.")

Doc 0 (Task 2's probe document, 'AirAsia Zest'):
  gold=22  baseline=21
  precision=0.810  recall=0.773  f1=0.791

Task 2 reported '17/22 (77%) match' for this document — that figure is recall (matched / gold), not precision or F1.


### Step 8: Distribution across documents

In [9]:
results_df[["precision", "recall", "f1"]].describe()

,precision,recall,f1
count,25.000000,25.000000,25.000000
mean,0.795067,0.792252,0.790422
std,0.112452,0.121512,0.105618
min,0.473684,0.571429,0.529412
25%,0.750000,0.714286,0.727273
50%,0.800000,0.800000,0.800000
75%,0.875000,0.894737,0.857143
max,1.000000,1.000000,1.000000


## Step 9: Findings write-up

**Setup**
- Sample: 25 documents from DocRED's `train_annotated` split (doc 0 = Task 2's original "AirAsia Zest" probe document, plus 24 seeded-random others).
- Pipeline: spaCy `en_core_web_sm` NER vs. DocRED gold entities (all mention aliases, normalized), scored as real precision/recall/F1 (TP/FP/FN) per document, then aggregated both micro (pooled counts) and macro (mean of per-doc scores).

**Results**
- Aggregate: **micro P=0.788, R=0.788, F1=0.788**; **macro P=0.795, R=0.792, F1=0.790** — the two aggregation methods land within 0.01 of each other, which makes sense given the sampled documents are fairly uniform in size (10–34 gold entities, no single document dominates the pooled counts).
- Precision and recall are almost exactly balanced in aggregate (micro P = micro R = 0.788), meaning spaCy isn't systematically biased toward over- or under-extraction on this task *on average* — but that balance hides real per-document swings (precision ranges 0.47–1.00, recall 0.57–1.00).
- Real variance across documents: F1 std = 0.106, ranging from **0.53** (doc 1723, "Victorian Gay and Lesbian Rights Lobby" — precision only 0.47, spaCy over-extracted with 10 false positives out of 19 predicted entities) to a perfect **1.00** (doc 1027, "President of Harvard University" — a short document, 11/11 gold entities exactly matched with zero false positives).
- **Cross-check against Task 2:** doc 0's recall here is 0.773 (17/22), matching Task 2's reported "77% match" exactly — confirming the harness correctly reproduces Task 2's original result. But Task 2's 77% was actually *recall*, not precision (0.810) or F1 (0.791); it never separated those three, since it only ever looked at "matched vs. gold," not spaCy's false positives.
- Doc 0's F1 (0.791) turned out to be very close to the 25-document aggregate F1 (0.788–0.790) — the original single-document estimate was, in hindsight, fairly representative of the average. That's a fortunate coincidence, not a validation of the one-document methodology: the ±0.11 std shows individual documents swing widely enough that no single probe document could be trusted to generalize without this kind of aggregate check.

**Open next steps**
1. Dig into the worst-performing document (doc 1723, F1=0.53) as a concrete case study — is the low precision a genuine spaCy limitation (e.g. splitting or merging entity spans differently than DocRED's annotation convention) or another artifact of the comparison harness itself (as Task 2 found twice already)?
2. Scale further (e.g. 100+ documents) now that the harness is verified, to get a tighter confidence interval on the aggregate F1 and confirm the micro/macro agreement holds at larger scale.
3. Re-run this exact harness against a better extractor (e.g. an LLM-prompted extractor) or a merged/domain-adapted model (Task 4) to use it as the "repeatable metric to compare future extraction/merging changes against" that Task 5 was meant to produce.
4. Extend to relation extraction precision/recall/F1 (still not attempted since Task 2), using the same TP/FP/FN scoring pattern established here.

**Conclusion:** Task 5's harness reproduces Task 2's original number exactly (confirming correctness) while revealing what that number actually was (recall, not F1) and how much it could have varied by chance (±0.11 F1 std across documents). The project now has a real, reusable precision/recall/F1 harness — with micro and macro aggregation — to score any future extraction or merging change against, rather than relying on single-document impressions.